# Training Script for Municipality of Sampaloc Quezon Chatbot System
**Architecture:** User Input → Preprocessing → Time Waster Detection → Sentiment Analysis → RAG Retrieval → FLAN-T5 Generation

## Import Libraries

In [1]:
import json
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, T5TokenizerFast, T5ForConditionalGeneration
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import os

c:\Users\stsim\OneDrive\Desktop\GOVERNMENT CHATBOT\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Citizen Charter Data

In [2]:
print("Loading citizen charter data...")
with open('../2. dataset preparation/citizens_charter_2025_sampaloc_quezon.json', 'r', encoding='utf-8') as f:
    charter_data = json.load(f)

print(f"Municipality: {charter_data['municipality']}, {charter_data['province']}")
print(f"Total Services: {charter_data['total_services']}")

Loading citizen charter data...
Municipality: Sampaloc, Quezon
Total Services: 109


In [3]:
# Process services into documents for RAG
documents = []
for service in charter_data['services']:
    # Extract requirements text
    requirements_text = ' '.join([req.get('requirement', '')[:100] for req in service.get('requirements', [])[:5]])
    
    # Extract fees from process flow
    fees_list = []
    for step in service.get('process_flow', []):
        fee = step.get('fees', '')
        if fee and fee.lower() not in ['none', '', 'n/a']:
            fees_list.append(fee)
    fees_text = ' '.join(fees_list[:3]) if fees_list else 'No fees'
    
    # Extract processing time
    time_list = [step.get('processing_time', '') for step in service.get('process_flow', []) if step.get('processing_time')]
    time_text = ' '.join(time_list[:3]) if time_list else service.get('total_processing_time', 'Not specified')
    
    # Create comprehensive text for embedding
    doc = {
        'service_id': service.get('service_id', ''),
        'service_name': service.get('service_name', ''),
        'office': service.get('office', ''),
        'classification': service.get('classification', ''),
        'transaction_type': service.get('transaction_type', ''),
        'who_may_avail': service.get('who_may_avail', ''),
        'description': service.get('description', ''),
        'requirements': service.get('requirements', []),
        'process_flow': service.get('process_flow', []),
        'requirements_text': requirements_text,
        'fees_text': fees_text,
        'time_text': time_text,
        'text': f"{service.get('service_name', '')}. Office: {service.get('office', '')}. Classification: {service.get('classification', '')}. Who may avail: {service.get('who_may_avail', '')}. Description: {service.get('description', '')}. Requirements: {requirements_text}. Fees: {fees_text}. Processing time: {time_text}"
    }
    documents.append(doc)

print(f"\nProcessed {len(documents)} services into documents")
print(f"Sample document keys: {list(documents[0].keys())}")


Processed 109 services into documents
Sample document keys: ['service_id', 'service_name', 'office', 'classification', 'transaction_type', 'who_may_avail', 'description', 'requirements', 'process_flow', 'requirements_text', 'fees_text', 'time_text', 'text']


## 2. Train Time Waster Detection

In [4]:
print("Training Time Waster Detection...")

time_waster_data = [
    ("How do I get a business permit?", 0),
    ("What are the requirements for marriage license?", 0),
    ("Where is the municipal office located?", 0),
    ("How much is the fee for building permit?", 0),
    ("What documents do I need for tax declaration?", 0),
    ("What is the process for birth certificate?", 0),
    ("How long does it take to get a permit?", 0),
    ("hello", 1),
    ("hi there", 1),
    ("test", 1),
    ("asdfgh", 1),
    ("lol", 1),
    ("hahaha", 1),
    ("???", 1),
]

X_tw = [x[0] for x in time_waster_data]
y_tw = [x[1] for x in time_waster_data]

tfidf = TfidfVectorizer(max_features=100)
X_tw_vec = tfidf.fit_transform(X_tw)

tw_classifier = LogisticRegression()
tw_classifier.fit(X_tw_vec, y_tw)

os.makedirs('models', exist_ok=True)
with open('models/time_waster_tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
with open('models/time_waster_classifier.pkl', 'wb') as f:
    pickle.dump(tw_classifier, f)

print("✓ Time waster detection model saved")

Training Time Waster Detection...
✓ Time waster detection model saved


## 3. Load Sentiment Analysis Model

In [5]:
print("Loading Sentiment Analysis Model...")

sentiment_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
sentiment_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

sentiment_model.save_pretrained('models/sentiment_model')
sentiment_tokenizer.save_pretrained('models/sentiment_tokenizer')

print("✓ Sentiment analysis model saved")

Loading Sentiment Analysis Model...


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.15s/it]

✓ Sentiment analysis model saved


## 4. Build RAG Index (FAISS)

In [6]:
print("Building RAG Index...")

embedder = SentenceTransformer('all-MiniLM-L6-v2')

doc_texts = [doc['text'] for doc in documents]
embeddings = embedder.encode(doc_texts, show_progress_bar=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))

faiss.write_index(index, 'models/faiss_index.bin')
with open('models/documents.json', 'w', encoding='utf-8') as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print(f"✓ FAISS index created with {len(documents)} documents")

Building RAG Index...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 352.91it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 4/4 [00:05<00:00,  1.25s/it]

✓ FAISS index created with 109 documents


## 5. Prepare FLAN-T5 Training Data

In [7]:
print("Preparing FLAN-T5 Training Data...")

training_data = []

for doc in documents:
    service_name = doc['service_name']
    office = doc['office']
    
    # General service info
    training_data.append({
        'input': f"What is {service_name}?",
        'output': f"{service_name} is a service provided by {office}. {doc['who_may_avail']} may avail this service. {doc['description']}"
    })
    
    # Requirements
    if doc['requirements']:
        req_list = [r.get('requirement', '') for r in doc['requirements'][:5]]
        req_text = '; '.join([r for r in req_list if r])
        training_data.append({
            'input': f"What are the requirements for {service_name}?",
            'output': f"The requirements for {service_name} include: {req_text}"
        })
    
    # Fees
    if doc['fees_text'] and doc['fees_text'] != 'No fees':
        training_data.append({
            'input': f"How much is the fee for {service_name}?",
            'output': f"The fees for {service_name} are: {doc['fees_text']}"
        })
    
    # Processing time
    if doc['time_text']:
        training_data.append({
            'input': f"How long does {service_name} take?",
            'output': f"The processing time for {service_name} is: {doc['time_text']}"
        })
    
    # Office location
    training_data.append({
        'input': f"Where can I get {service_name}?",
        'output': f"You can get {service_name} at the {office}."
    })
    
    # Who may avail
    if doc['who_may_avail']:
        training_data.append({
            'input': f"Who can avail {service_name}?",
            'output': f"{doc['who_may_avail']} can avail {service_name}."
        })

df = pd.DataFrame(training_data)
df.to_csv('models/training_data.csv', index=False)

print(f"Generated {len(training_data)} training examples")
print(f"Sample training example:")
print(f"  Input: {training_data[0]['input']}")
print(f"  Output: {training_data[0]['output'][:100]}...")

Preparing FLAN-T5 Training Data...
Generated 596 training examples
Sample training example:
  Input: What is Processing of Vouchers?
  Output: Processing of Vouchers is a service provided by Municipal Accounting Office. Officials and Employees...


## 6. Load FLAN-T5 Model

In [8]:
print("Loading FLAN-T5 Model...")

model_name = "google/flan-t5-base"
tokenizer = T5TokenizerFast.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

model.save_pretrained('models/flan_t5_model')
tokenizer.save_pretrained('models/flan_t5_tokenizer')

print("✓ FLAN-T5 base model saved")

Loading FLAN-T5 Model...


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 362.99it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Writing model shards: 100%|██████████| 1/1 [00:30<00:00, 30.37s/it]

✓ FLAN-T5 base model saved


## 7. Save Embedder Model

In [9]:
print("Saving Sentence Transformer model...")
embedder.save('models/sentence_transformer')
print("✓ Sentence Transformer model saved")

Saving Sentence Transformer model...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.40it/s]

✓ Sentence Transformer model saved


## 8. Training Complete!

In [10]:
print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)
print("\nChatbot System Summary:")
print(f"  • Municipality: {charter_data['municipality']}, {charter_data['province']}")
print(f"  • Total Services: {len(documents)}")
print(f"  • Training Examples: {len(training_data)}")
print("\nModels saved:")
print("  ✓ Time Waster Detection: models/time_waster_*.pkl")
print("  ✓ Sentiment Analysis: models/sentiment_model/")
print("  ✓ RAG Index: models/faiss_index.bin")
print("  ✓ Documents: models/documents.json")
print("  ✓ FLAN-T5: models/flan_t5_model/")
print("  ✓ Sentence Transformer: models/sentence_transformer/")
print("  ✓ Training Data: models/training_data.csv")
print("\nThe chatbot can now answer questions about:")
print("  • Service descriptions and processes")
print("  • Requirements for each service")
print("  • Fees and charges")
print("  • Processing times")
print("  • Office locations")
print("  • Who can avail services")
print("="*60)


TRAINING COMPLETE!

Chatbot System Summary:
  • Municipality: Sampaloc, Quezon
  • Total Services: 109
  • Training Examples: 596

Models saved:
  ✓ Time Waster Detection: models/time_waster_*.pkl
  ✓ Sentiment Analysis: models/sentiment_model/
  ✓ RAG Index: models/faiss_index.bin
  ✓ Documents: models/documents.json
  ✓ FLAN-T5: models/flan_t5_model/
  ✓ Sentence Transformer: models/sentence_transformer/
  ✓ Training Data: models/training_data.csv

The chatbot can now answer questions about:
  • Service descriptions and processes
  • Requirements for each service
  • Fees and charges
  • Processing times
  • Office locations
  • Who can avail services
